# Prompt Engineering Assignment: Cafe Order Processing System

Converts natural-language cafe orders into strict JSON using `gpt-4o-mini`.

Staff ask *"What would you like to have today?"* and the customer's reply is converted into:

```json
{
  "items": [
    {"name": "Americano", "quantity": 2, "size": "regular"},
    {"name": "Fries", "quantity": 1, "size": "large"},
    {"name": "Hamburger", "quantity": 3, "size": "regular"}
  ],
  "total_items": 6
}
```

**Approach:** a short system prompt for interpreting the order, plus a JSON schema passed through `response_format` (Structured Outputs, strict mode). The schema guarantees the reply always has exactly this shape, so the prompt doesn't need to describe the format.

In [1]:
# !pip install openai
import json
import os
from getpass import getpass

from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

Enter your OpenAI API key: ··········


## The JSON schema

`ORDER_SCHEMA` is plain JSON Schema matching the required output exactly. The OpenAI-specific settings (`name` and `strict`) are added where the API is called, in `process_order`.

In strict mode every property must be listed in `required`, and every object needs `"additionalProperties": False`. The `size` enum restricts sizes to the three allowed values.

In [2]:
ORDER_SCHEMA = {
    "type": "object",
    "properties": {
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "quantity": {"type": "integer"},
                    "size": {"type": "string", "enum": ["small", "regular", "large"]},
                },
                "required": ["name", "quantity", "size"],
                "additionalProperties": False,
            },
        },
        "total_items": {"type": "integer"},
    },
    "required": ["items", "total_items"],
    "additionalProperties": False,
}

## The system prompt

The schema handles the format, so the prompt only explains how to read an order.

The required output has no field for customisations, so variants go into the item name: *"make one of them decaf"* becomes a separate **Decaf Iced Latte**, and *"burger meal with coke"* becomes **Burger Meal with Coke**. This keeps the brief's schema unchanged without losing information.

In [3]:
SYSTEM_PROMPT = """You process orders at a cafe. Staff ask the customer "What would you like to have today?" and you receive the reply. Extract every item ordered.

- name: the item in singular form and Title Case ("hamburgers" -> "Hamburger", "iced lattes" -> "Iced Latte").
- quantity: convert words and shorthand to a number ("a", "one" -> 1; "two" -> 2; "a couple of" -> 2; "2x", "x2" -> 2). Default 1.
- size: "small", "regular" or "large". Map "medium" to "regular". Default "regular" when no size is given.
- A variant becomes part of the name. If only some units differ, split them into separate entries: "three iced lattes, make one of them decaf" -> Iced Latte x2 and Decaf Iced Latte x1.
- A meal keeps its chosen drink or side in the name: "burger meal with coke" -> "Burger Meal with Coke". Anything ordered "extra" or "on the side" is a separate item.
- total_items: the sum of all quantities.
- Ignore greetings and filler. If nothing is ordered, return no items and total_items 0."""

## Order processing

Follows the example code structure, with the schema added to `response_format`. `total_items` is recomputed in code as a safety check, since arithmetic is where a model is most likely to slip.

In [4]:
def process_order(customer_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": customer_input},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "cafe_order", "strict": True, "schema": ORDER_SCHEMA},
        },
        temperature=0.1,  # Low temperature for consistent output
    )
    order = json.loads(response.choices[0].message.content)
    order["total_items"] = sum(item["quantity"] for item in order["items"])
    return order

## Test with various inputs

The four orders from the brief, followed by extra cases. The comment beside each extra case is the expected result.

In [5]:
orders = [
    # From the brief
    "2x Americano, 1 large fries and 3 hamburger",
    "I'll have a cappuccino and two croissants please",
    "Can I get three iced lattes, make one of them decaf",
    "One burger meal with coke, and an extra order of fries",
    # Additional test cases
    "A medium flat white and a couple of blueberry muffins",   # Flat White x1 regular, Blueberry Muffin x2 -> 3
    "x3 espresso and a bagel",                                 # Espresso x3, Bagel x1 -> 4
    "Two small hot chocolates and a large hot chocolate",      # Hot Chocolate x2 small, x1 large -> 3
    "Four lattes, two of them with oat milk",                  # Latte x2, Oat Milk Latte x2 -> 4
    "A large iced latte... actually, make that two",           # Iced Latte x2 large -> 2
    "Two kopi and one kaya toast",                             # Kopi x2, Kaya Toast x1 -> 3
    "Hi, how long is the wait today?",                         # no items -> 0
]

for order in orders:
    print(f"Input: {order}")
    print(f"Output: {json.dumps(process_order(order), indent=2)}")
    print()

Input: 2x Americano, 1 large fries and 3 hamburger
Output: {
  "items": [
    {
      "name": "Americano",
      "quantity": 2,
      "size": "regular"
    },
    {
      "name": "Fries",
      "quantity": 1,
      "size": "large"
    },
    {
      "name": "Hamburger",
      "quantity": 3,
      "size": "regular"
    }
  ],
  "total_items": 6
}

Input: I'll have a cappuccino and two croissants please
Output: {
  "items": [
    {
      "name": "Cappuccino",
      "quantity": 1,
      "size": "regular"
    },
    {
      "name": "Croissant",
      "quantity": 2,
      "size": "regular"
    }
  ],
  "total_items": 3
}

Input: Can I get three iced lattes, make one of them decaf
Output: {
  "items": [
    {
      "name": "Iced Latte",
      "quantity": 2,
      "size": "regular"
    },
    {
      "name": "Decaf Iced Latte",
      "quantity": 1,
      "size": "regular"
    }
  ],
  "total_items": 3
}

Input: One burger meal with coke, and an extra order of fries
Output: {
  "items": [
    